# Mino-IA — Fine-tuning LoRA de Mistral 7B

Notebook prêt pour **Google Colab** et **Kaggle**.

Il entraîne l'adaptateur de **rédaction** du compte rendu. Le moteur
mendélien de Mino-IA reste la source de vérité : le LLM apprend à
reformuler les faits déjà calculés, jamais à recalculer une probabilité.

> **GPU obligatoire.** T4 16 Go minimum ; A100 40 Go recommandé pour le
> mode `full`.
>
> **Durée indicative** : mode `smoke` (200 exemples, 1 époque) quelques
> dizaines de minutes sur T4 ; mode `full` (4 000 exemples, 3 époques)
> plusieurs heures, avec un risque de déconnexion sur Colab gratuit.

## 1. Prérequis

- **Colab** : *Exécution → Modifier le type d'exécution → T4 GPU* (ou A100).
- **Kaggle** : *Settings → Accelerator → GPU T4 x2 / P100 / A100*.
- Un compte Hugging Face est recommandé pour éviter les limites de débit.
  Mistral 7B Instruct v0.3 est public, mais un token reste utile.
- Environ 20 Go d'espace disque.
- Sur Colab, monter Google Drive est conseillé pour conserver l'adaptateur.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print("Python :", sys.version.split()[0])
print("Système :", platform.platform())
print("CPU :", os.cpu_count(), "cœurs")
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
    print(f"RAM : {ram_gb:.1f} Go")
except (ValueError, OSError):
    pass

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("⚠️ Aucun GPU détecté. Active un GPU avant de continuer.")

try:
    import torch

    print("CUDA disponible :", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU :", props.name)
        print(f"VRAM : {props.total_memory / 1024**3:.1f} Go")
except Exception as exc:
    print("Torch non disponible :", exc)

## 2. Récupérer le projet

Le notebook clone le dépôt public. Sur Kaggle, le projet est placé dans
`/tmp` pour ne pas polluer l'archive de sortie ; seul l'adaptateur final
est écrit dans `/kaggle/working`.

In [ ]:
REPO_URL = "https://github.com/hermosdessossa-commits/Mino-IA-v2.1.0.git"

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    PROJECT_DIR = Path("/tmp/Mino-IA-v2.1.0")
    EXPORT_ROOT = Path("/kaggle/working")
elif Path("/content").exists():
    PROJECT_DIR = Path("/content/Mino-IA-v2.1.0")
    EXPORT_ROOT = Path("/content")
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_DIR = next(
        (path for path in candidates if (path / "pyproject.toml").exists()),
        Path.cwd() / "Mino-IA-v2.1.0",
    )
    EXPORT_ROOT = PROJECT_DIR.parent

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
elif (PROJECT_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=False,
    )

os.chdir(PROJECT_DIR)
print("Projet :", PROJECT_DIR)
print(
    subprocess.run(
        ["git", "log", "--oneline", "-1"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)

## 3. Installer les dépendances

L'installation prend quelques minutes. Sur Colab, le PyTorch CUDA déjà
présent est conservé s'il satisfait `torch>=2.1,<3`.

In [ ]:
print("Installation des dépendances ML...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ml.txt"],
    check=True,
)

import accelerate
import bitsandbytes
import datasets
import peft
import torch
import transformers

print("torch        :", torch.__version__, "| CUDA :", torch.cuda.is_available())
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("accelerate   :", accelerate.__version__)
print("datasets     :", datasets.__version__)

## 4. Connexion Hugging Face (optionnelle)

Colab : colle un token avec le champ masqué. Kaggle : préfère
*Add-ons → Secrets* et crée un secret `HF_TOKEN`.

In [ ]:
import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    try:
        HF_TOKEN = getpass("Token Hugging Face (laisser vide si le modèle est public) : ").strip()
    except Exception:
        HF_TOKEN = ""

if HF_TOKEN:
    from huggingface_hub import login

    login(token=HF_TOKEN, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Connecté à Hugging Face.")
else:
    print("Pas de token fourni : téléchargement public uniquement.")

## 5. Vérifier le dataset

Le validateur recalcule les statuts et probabilités avec le moteur
déterministe. Aucune probabilité n'est inventée par le LLM.

In [ ]:
subprocess.run(
    [sys.executable, "training/finetuning_lora.py", "--check-dataset"],
    check=True,
)

## 6. Choisir le mode et les paramètres

- `smoke` : 200 paires, 1 époque, évaluation courte. Sert à vérifier que
  la chaîne fonctionne avant de lancer un entraînement long.
- `full` : 4 000 paires, 3 époques. Produit l'adaptateur destiné à une
  évaluation sérieuse.

Sur un GPU de moins de 18 Go, la longueur maximale est automatiquement
réduite à 1 024 tokens. En cas d'OOM, descendre à 768 ou 512.

In [ ]:
import torch

RUN_MODE = "smoke"  # "smoke" pour valider, "full" pour l'entraînement complet

MODEL = os.environ.get("MINO_MODEL", "mistralai/Mistral-7B-Instruct-v0.3")
OUTPUT_DIR = PROJECT_DIR / "model" / "lora-adapter"

BATCH_SIZE = 1
GRAD_ACCUM = 16
LORA_R = 16
LORA_ALPHA = 32
SEED = 42
GRADIENT_CHECKPOINTING = True

if RUN_MODE == "smoke":
    EPOCHS = 1
    MAX_TRAIN_SAMPLES = 200
    MAX_EVAL_SAMPLES = 10
    MAX_NEW_TOKENS = 128
    WARMUP_STEPS = 0
elif RUN_MODE == "full":
    EPOCHS = 3
    MAX_TRAIN_SAMPLES = 0
    MAX_EVAL_SAMPLES = 50
    MAX_NEW_TOKENS = 512
    WARMUP_STEPS = 50
else:
    raise ValueError("RUN_MODE doit valoir 'smoke' ou 'full'.")

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA requis pour ce notebook.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
MAX_SEQ_LENGTH = 2048
if vram_gb < 18:
    MAX_SEQ_LENGTH = 1024
elif vram_gb < 30:
    MAX_SEQ_LENGTH = 1536

if RUN_MODE == "full" and vram_gb < 18:
    MAX_EVAL_SAMPLES = 20

print(f"Mode             : {RUN_MODE}")
print(f"GPU / VRAM       : {torch.cuda.get_device_name(0)} / {vram_gb:.1f} Go")
print(f"Modèle           : {MODEL}")
print(f"Époques          : {EPOCHS}")
print(f"Paires max       : {MAX_TRAIN_SAMPLES or 'toutes'}")
print(f"Longueur max     : {MAX_SEQ_LENGTH}")
print(f"Batch effectif   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Sortie           : {OUTPUT_DIR}")

## 7. Lancer l'entraînement

Le script détecte CUDA, charge le modèle en 4 bits avec `bitsandbytes`,
applique LoRA, masque le prompt dans la loss et sauvegarde l'adaptateur.

In [ ]:
cmd = [
    sys.executable,
    "training/finetuning_lora.py",
    "--model",
    MODEL,
    "--train-csv",
    "data/dataset_synthetique.csv",
    "--test-csv",
    "data/dataset_test.csv",
    "--output-dir",
    str(OUTPUT_DIR),
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--gradient-accumulation",
    str(GRAD_ACCUM),
    "--max-seq-length",
    str(MAX_SEQ_LENGTH),
    "--max-new-tokens",
    str(MAX_NEW_TOKENS),
    "--max-train-samples",
    str(MAX_TRAIN_SAMPLES),
    "--max-eval-samples",
    str(MAX_EVAL_SAMPLES),
    "--lora-r",
    str(LORA_R),
    "--lora-alpha",
    str(LORA_ALPHA),
    "--warmup-steps",
    str(WARMUP_STEPS),
    "--logging-steps",
    "10",
    "--save-steps",
    "500",
    "--seed",
    str(SEED),
]
if GRADIENT_CHECKPOINTING:
    cmd.append("--gradient-checkpointing")

print("Commande :\n" + " ".join(cmd))
subprocess.run(cmd, check=True)

## 8. Consulter les métriques

Les métriques sont calculées sur le jeu de test avec le JSON produit par
le LLM. Une sortie non JSON est ignorée et ne remplace jamais le rapport
déterministe.

In [ ]:
import json

metadata_path = OUTPUT_DIR / "training_metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print(json.dumps(metadata, ensure_ascii=False, indent=2))
else:
    print("Métadonnées absentes :", metadata_path)

## 9. Exporter l'adaptateur

Seuls les fichiers utiles à l'inférence sont copiés : configuration et
poids LoRA, tokenizer et métadonnées. Les checkpoints d'entraînement
(`optimizer.pt`, `rng_state.pth`, etc.) sont exclus de l'archive.

In [ ]:
EXPORT_DIR = EXPORT_ROOT / "mino-ia-lora-adapter"
EXPORT_FILES = (
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
    "training_metadata.json",
)

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

copied = []
for name in EXPORT_FILES:
    source = OUTPUT_DIR / name
    if source.exists():
        shutil.copy2(source, EXPORT_DIR / name)
        copied.append(name)

print("Fichiers exportés :", ", ".join(copied))
print("Dossier           :", EXPORT_DIR)

archive_path = shutil.make_archive(
    str(EXPORT_ROOT / "mino-ia-lora-adapter"),
    "zip",
    root_dir=EXPORT_DIR,
)
print("Archive           :", archive_path)

In [ ]:
# Colab : copier aussi l'adaptateur sur Google Drive.
if Path("/content").exists():
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        drive_dir = Path("/content/drive/MyDrive/mino-ia/lora-adapter")
        if drive_dir.exists():
            shutil.rmtree(drive_dir)
        shutil.copytree(EXPORT_DIR, drive_dir)
        print("Copie Google Drive :", drive_dir)
    except Exception as exc:
        print("Google Drive non monté :", exc)

# Colab : proposer le téléchargement de l'archive.
if Path("/content").exists():
    try:
        from google.colab import files

        files.download(archive_path)
    except Exception as exc:
        print("Téléchargement navigateur indisponible :", exc)

## 10. Vérification optionnelle de l'adaptateur

Passer `RUN_ADAPTER_CHECK = True` pour recharger le modèle de base et
l'adaptateur, puis générer un compte rendu. Sur T4, cette étape prend
plusieurs minutes. Si la sortie n'est pas un JSON valide, le repli
déterministe est utilisé : c'est le comportement attendu.

In [ ]:
RUN_ADAPTER_CHECK = False

if RUN_ADAPTER_CHECK:
    from api.llm import LLMEngine
    from core.genetics import predict
    from core.prompts import build_user_prompt
    from core.report import build_recommendations, build_report

    engine = LLMEngine(
        MODEL,
        enabled=True,
        adapter_dir=OUTPUT_DIR,
        load_in_4bit=True,
        max_new_tokens=256,
    )
    if not engine.load():
        raise RuntimeError(engine.error)

    prediction = predict("drepanocytose", "A/S", "A/S")
    prompt = build_user_prompt(
        "drepanocytose",
        "A/S",
        "A/S",
        prediction=prediction,
        draft_report=build_report(prediction),
        draft_recommendations=build_recommendations(
            "drepanocytose",
            prediction.status,
            prediction.probability,
        ),
    )
    result = engine.generate_report(prompt)
    if result:
        print("Rapport LLM :\n", result["rapport"])
        print("\nRecommandations :\n", result["recommandations"])
    else:
        print("Sortie LLM inexploitable : repli déterministe conservé.")

## Dépannage

- **CUDA out of memory** : réduire `MAX_SEQ_LENGTH` à 768 ou 512,
  `MAX_TRAIN_SAMPLES` à 100, garder `BATCH_SIZE = 1` et
  `GRADIENT_CHECKPOINTING = True`. Utiliser une A100 si possible.
- **`bitsandbytes` introuvable ou erreur CUDA** : redémarrer le runtime
  après l'installation, puis vérifier que le GPU est bien activé.
- **Déconnexion Colab** : lancer d'abord `RUN_MODE = "smoke"`, conserver
  l'adaptateur sur Drive, puis utiliser Colab Pro/A100 ou Kaggle pour le
  mode `full`.
- **Modèle inaccessible** : accepter la licence sur Hugging Face et
  fournir `HF_TOKEN` via le champ masqué ou Kaggle Secrets.
- **Sortie non JSON** : le LLM ne remplace jamais les valeurs du moteur ;
  l'API renvoie le rapport déterministe.

> Rappel : un adaptateur entraîné sur des données synthétiques ne
> constitue pas un dispositif médical. Toute utilisation clinique exige
> une validation professionnelle et réglementaire.